# HDFC Bank Service Request Resolution Analytics

## Your Engineering Handover

You have joined the HDFC Bank Service Operations team as a Junior Python Developer. The operations lead spends hours switching between service request records, customer complaints, branch information, and request status updates to understand which cases are delayed, unresolved, or require immediate attention. You have been asked to build a reproducible service request resolution analysis workflow in this notebook.

Your deliverable is not an app or dashboard. It is a trustworthy set of cleaned data tables, a request-level investigation report, a branch SLA performance summary, a prioritized service resolution queue, and a concise evidence-based handover to Service Operations.

How to work: Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

Input files (only three): `service_requests.csv` · `request_status_updates.xlsx` · `service_evidence.json` Data: Fictional training records; not actual HDFC Bank customer records.

> Important: The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks. Do not upload real banking or customer information to public AI tools.

---
**Project Summary:** Banking customer-service analytics in Python and Pandas. The notebook loads request, status-update, customer, branch, ticket and complaint data, cleans it, measures SLA breaches, builds one row per request, assigns a transparent High/Medium/Low priority, and exports three reports.

**GitHub Repo:** `PASTE-YOUR-GITHUB-LINK-HERE`  |  **Author:** `YOUR NAME`

## Important Note: Guidelines for Using AI Tools

You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with specific code snippets or individual problems.

However, using AI to generate the complete project solution in one go is not permitted.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

Remember: The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## Workspace setup · Data intake

Run the setup cell and upload the three input files when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

Link to Dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['service_requests.csv', 'service_evidence.json', 'request_status_updates.xlsx']
DATASET_ZIP = 'hdfc_service_request_project_datasets.zip'
missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing and (DATA_DIR/DATASET_ZIP).is_file():
    with zipfile.ZipFile(DATA_DIR/DATASET_ZIP) as archive:
        archive.extractall(DATA_DIR)
    missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing and DATA_DIR.as_posix() == '/content':
    from google.colab import files
    print('Upload the dataset ZIP or all three files (CSV, JSON and XLSX):')
    files.upload()
    if (DATA_DIR/DATASET_ZIP).is_file():
        with zipfile.ZipFile(DATA_DIR/DATASET_ZIP) as archive:
            archive.extractall(DATA_DIR)
    missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing required HDFC files in {DATA_DIR}: {missing}')
print('HDFC files ready:', len(REQUIRED_FILES), '| Directory:', DATA_DIR)

Upload the dataset ZIP or all three files (CSV, JSON and XLSX):


Saving request_status_updates.xlsx to request_status_updates.xlsx
Saving service_evidence.json to service_evidence.json
Saving service_requests.csv to service_requests.csv
HDFC files ready: 3 | Directory: /content


## Milestone 01 · Establish the service request baseline

**From the team:** The service operations lead needs a trusted intake snapshot before any service-level calculations.

**Your responsibility:** Load the request CSV and inspect raw rows, unique request IDs, customers, categories, missing values, and data types.

**Handover evidence:** `requests_df` as the raw request DataFrame; show row count, distinct IDs, schema, missingness and a preview.

In [ ]:
# Milestone 01: Establish the service request baseline
# Task 01.1: Read the input CSV with Pandas into requests_df.
# Task 01.2: Print total rows, unique request IDs and customer IDs.
# Task 01.3: Inspect dtypes, null counts and the first three records.
# Implement your solution below and display the required handover evidence.

requests_df = pd.read_csv(DATA_DIR / 'service_requests.csv')

print('Total rows           :', len(requests_df))
print('Unique request IDs   :', requests_df['request_id'].nunique())
print('Unique customer IDs  :', requests_df['customer_id'].nunique())
print('Duplicate request IDs:', len(requests_df) - requests_df['request_id'].nunique(), '(extra rows)')

print('\nData types:')
print(requests_df.dtypes)
print('\nNull counts:')
print(requests_df.isna().sum())

print('\nRaw category labels (note the spacing/case differences):')
print(requests_df['request_category'].value_counts())

requests_df.head(3)

Total rows           : 5000
Unique request IDs   : 4992
Unique customer IDs  : 1189
Duplicate request IDs: 8 (extra rows)

Data types:
request_id              object
customer_id             object
branch_id               object
request_created_date    object
request_created_time    object
request_category        object
request_subcategory     object
channel                 object
request_status          object
sla_hours                int64
assigned_team           object
reported_amount         object
last_updated_date       object
resolved_date           object
dtype: object

Null counts:
request_id                 0
customer_id                0
branch_id                  0
request_created_date       0
request_created_time       0
request_category           0
request_subcategory        0
channel                    0
request_status             0
sla_hours                  0
assigned_team              0
reported_amount         1669
last_updated_date          0
resolved_date           26

,request_id,customer_id,branch_id,request_created_date,request_created_time,request_category,request_subcategory,channel,request_status,sla_hours,assigned_team,reported_amount,last_updated_date,resolved_date
0,REQ0000009,CUST001111,BR0566,2026-04-27,04:20:00,Complaint,Service Delay,Branch,In Progress,72,Customer Grievance,240247.88,2026-04-27,NaN
1,REQ0000010,CUST000649,BR0980,2026-05-17,22:31:00,Card,Failed ATM Withdrawal,WhatsApp,Pending Customer,36,Cards Operations,24383.2,2026-05-18,NaN
2,REQ0000011,CUST000817,BR0045,2026-06-19,04:47:00,Digital Banking,NetBanking Login,Email,Closed,24,Digital Banking Support,193459.78,2026-06-19,2026-06-19


## Milestone 02 · Onboard branch and service evidence

**From the team:** The branch desk, ticket desk, and escalation team each use a different export format.

**Your responsibility:** Load customers, branches, support tickets, status-update spreadsheet and complaint JSON. Inspect counts, statuses and relationships. Read contextual TXT notes.

**Handover evidence:** `customers_df`, `branches_df`, `tickets_df`, `updates_df`, `complaints_df` and a short input QA summary.

**Input packaging:** Three files only: the JSON holds four logically separate entity/event collections, and the XLSX holds status updates.

In [ ]:
# Milestone 02: Onboard branch and service evidence
# Task 02.1: Open service_evidence.json using json.load. Verify it contains exactly four named collections: customers, branches, support_tickets, customer_complaints.
# Task 02.2: Create customers_df, branches_df, tickets_df, and complaints_df as separate Pandas DataFrames from these JSON collections.
# Task 02.3: Read request_status_updates.xlsx into updates_df with Pandas.
# Task 02.4: Display row counts, missing values, ticket statuses, and complaint statuses.
# Expected output: the five named DataFrames for downstream milestones.

with open(DATA_DIR / 'service_evidence.json') as f:
    evidence = json.load(f)

expected_collections = {'customers', 'branches', 'support_tickets', 'customer_complaints'}
assert set(evidence.keys()) == expected_collections, f'Unexpected JSON collections: {set(evidence.keys())}'
print('JSON collections found:', sorted(evidence.keys()))

customers_df  = pd.DataFrame(evidence['customers'])
branches_df   = pd.DataFrame(evidence['branches'])
tickets_df    = pd.DataFrame(evidence['support_tickets'])
complaints_df = pd.DataFrame(evidence['customer_complaints'])
updates_df    = pd.read_excel(DATA_DIR / 'request_status_updates.xlsx')

# Optional context notes: the three-file package does not include the TXT file, so only read it if present
notes_path = DATA_DIR / 'status_notes.txt'
if notes_path.is_file():
    print('\nContext notes (first 3 lines):')
    print(''.join(open(notes_path).readlines()[:3]))
else:
    print('\nstatus_notes.txt not supplied in this package (context notes only, not needed for calculations).')

input_qa = pd.DataFrame({
    'table': ['customers_df', 'branches_df', 'tickets_df', 'updates_df', 'complaints_df'],
    'rows': [len(customers_df), len(branches_df), len(tickets_df), len(updates_df), len(complaints_df)],
    'missing_cells': [int(d.isna().sum().sum()) for d in [customers_df, branches_df, tickets_df, updates_df, complaints_df]],
})
display(input_qa)
print('Ticket statuses (raw):'); print(tickets_df['ticket_status'].value_counts().to_dict())
print('\nComplaint statuses (raw):'); print(complaints_df['complaint_status'].value_counts().to_dict())

JSON collections found: ['branches', 'customer_complaints', 'customers', 'support_tickets']

status_notes.txt not supplied in this package (context notes only, not needed for calculations).


,table,rows,missing_cells
0,customers_df,1200,0
1,branches_df,1000,0
2,tickets_df,1500,499
3,updates_df,1300,0
4,complaints_df,1500,0


Ticket statuses (raw):
{'Open': 262, 'Reopened': 250, 'Escalated': 245, 'Closed': 241, 'Resolved': 235, 'Pending': 231, 'resolved': 9, 'reopened': 8, 'open': 6, 'closed': 5, 'pending': 5, 'escalated': 3}

Complaint statuses (raw):
{'Reopened': 263, 'Resolved': 251, 'Escalated': 248, 'Closed': 238, 'In Review': 235, 'Open': 231, 'reopened': 11, 'closed': 7, 'in review': 6, 'resolved': 4, 'escalated': 4, 'open': 2}


## Milestone 03 · Prepare dependable service requests

**From the team:** Duplicate IDs and inconsistent status labels can inflate SLA breaches and obscure unresolved requests.

**Your responsibility:** Keep one record per request ID, normalize category/channel/status, parse creation and resolution dates/times, convert SLA and monetary values to numeric, and identify invalid closure dates.

**Handover evidence:** `requests_clean`, one row per `request_id`, with `created_at`, `resolved_at`, `is_resolved`, `request_date_issue`.

In [ ]:
# Milestone 03: Prepare dependable service requests
# Task 03.1: Remove duplicate request IDs, retaining one documented record.
# Task 03.2: Normalize status/category/channel labels and convert SLA and amounts to numeric.
# Task 03.3: Parse created_at, resolved_at and last update; compute is_resolved and request_date_issue.
# Implement your solution below and display the required handover evidence.

def canonical_labels(series):
    """Make ' loan ', 'LOAN' and 'loan' one label: the most common spelling in the data."""
    s = series.astype('string')
    key = s.str.strip().str.lower().str.replace('_', ' ', regex=False).str.replace(r'\s+', ' ', regex=True)
    best = (pd.DataFrame({'key': key, 'raw': s.str.strip()}).dropna()
              .groupby('key')['raw'].agg(lambda x: x.value_counts().index[0]))
    return key.map(best)

requests_clean = requests_df.copy()

# IDs: trim spaces, upper-case
for col in ['request_id', 'customer_id', 'branch_id']:
    requests_clean[col] = requests_clean[col].astype('string').str.strip().str.upper()

# Task 03.1 - duplicates: keep the FIRST row (the original record); the later copies carry conflicting statuses
dup_rows = requests_clean[requests_clean['request_id'].duplicated(keep='first')]
print('Duplicate request rows removed:', len(dup_rows), '-> kept the first (original) record of each request_id')
requests_clean = requests_clean.drop_duplicates(subset='request_id', keep='first').reset_index(drop=True)

# Task 03.2 - labels and numbers
for col in ['request_category', 'request_subcategory', 'channel', 'request_status', 'assigned_team']:
    requests_clean[col] = canonical_labels(requests_clean[col])
requests_clean['sla_hours'] = pd.to_numeric(requests_clean['sla_hours'], errors='coerce')
requests_clean['reported_amount'] = pd.to_numeric(requests_clean['reported_amount'], errors='coerce')  # 'amount_pending' becomes blank

# Task 03.3 - dates (invalid values like 2026-02-30 or 'not_available' become NaT, never deleted rows)
created_date = pd.to_datetime(requests_clean['request_created_date'], format='%Y-%m-%d', errors='coerce')
created_time = pd.to_timedelta(requests_clean['request_created_time'], errors='coerce')
requests_clean['created_at'] = created_date + created_time
resolved_date = pd.to_datetime(requests_clean['resolved_date'], format='%Y-%m-%d', errors='coerce')
# the data holds only a date for resolution, so a request counts as resolved at the END of that day
requests_clean['resolved_at'] = resolved_date + pd.Timedelta(hours=23, minutes=59, seconds=59)
requests_clean['last_updated_at'] = pd.to_datetime(requests_clean['last_updated_date'], format='%Y-%m-%d', errors='coerce')

requests_clean['is_resolved'] = requests_clean['request_status'].isin(['Resolved', 'Closed'])

# request_date_issue: a value was present but unusable, or the closure date does not make sense
requests_clean['request_date_issue'] = (
    requests_clean['created_at'].isna()
    | (requests_clean['resolved_date'].notna() & resolved_date.isna())      # e.g. 'not_available'
    | (requests_clean['resolved_at'] < requests_clean['created_at'])        # resolved before it was created
    | (requests_clean['is_resolved'] & requests_clean['resolved_at'].isna())# closed but no closure date
)

print('Rows now:', len(requests_clean), '| unique request IDs:', requests_clean['request_id'].is_unique)
print('Requests with a date issue (flagged, kept):', int(requests_clean['request_date_issue'].sum()))
print('Resolved / Closed:', int(requests_clean['is_resolved'].sum()), '| Unresolved:', int((~requests_clean['is_resolved']).sum()))
print('Non-numeric amounts turned blank:', int(requests_clean['reported_amount'].isna().sum() - requests_df['reported_amount'].isna().sum()))
print('Statuses:', sorted(requests_clean['request_status'].dropna().unique()))
requests_clean[['request_id', 'request_category', 'request_status', 'sla_hours', 'created_at', 'resolved_at', 'is_resolved', 'request_date_issue']].head()

Duplicate request rows removed: 8 -> kept the first (original) record of each request_id
Rows now: 4992 | unique request IDs: True
Requests with a date issue (flagged, kept): 11
Resolved / Closed: 2360 | Unresolved: 2632
Non-numeric amounts turned blank: 2
Statuses: ['Closed', 'Escalated', 'In Progress', 'Open', 'Pending Customer', 'Resolved']


,request_id,request_category,request_status,sla_hours,created_at,resolved_at,is_resolved,request_date_issue
0,REQ0000009,Complaint,In Progress,72,2026-04-27 04:20:00,NaT,False,False
1,REQ0000010,Card,Pending Customer,36,2026-05-17 22:31:00,NaT,False,False
2,REQ0000011,Digital Banking,Closed,24,2026-06-19 04:47:00,2026-06-19 23:59:59,True,False
3,REQ0000012,Complaint,In Progress,72,2026-04-20 03:36:00,NaT,False,False
4,REQ0000013,Account,Closed,48,2026-06-13 07:18:00,2026-06-16 23:59:59,True,False


## Milestone 04 · Standardize customer and event records

**From the team:** A request can legitimately have many updates, tickets and complaints; dropping repeated request IDs would erase evidence.

**Your responsibility:** Dedupe only by each table's own unique ID, normalize event statuses and flags, parse dates, and check unmatched foreign keys.

**Handover evidence:** `customers_clean`, `branches_clean`, `tickets_clean`, `updates_clean`, `complaints_clean`, with retained distinct events.

In [ ]:
# Milestone 04: Standardize customer and event records
# Task 04.1: Dedupe each dimension/event table using its own primary identifier.
# Task 04.2: Normalize customer segment, event status and boolean escalation/reassignment flags.
# Task 04.3: Parse complaint/ticket/update dates and report broken foreign key links.
# Implement your solution below and display the required handover evidence.

def clean_ids(frame, cols):
    for c in cols:
        frame[c] = frame[c].astype('string').str.strip().str.upper()
    return frame

def to_bool(series):
    return series.astype('string').str.strip().str.lower().isin(['yes', 'y', 'true', '1'])

# Task 04.1 + 04.2: each table is deduplicated on its OWN primary key only
customers_clean = clean_ids(customers_df.copy(), ['customer_id', 'preferred_branch_id'])
customers_clean = customers_clean.drop_duplicates('customer_id').reset_index(drop=True)
customers_clean['customer_segment'] = canonical_labels(customers_clean['customer_segment'])
customers_clean['relationship_age_days'] = pd.to_numeric(customers_clean['relationship_age_days'], errors='coerce')

branches_clean = clean_ids(branches_df.copy(), ['branch_id'])
branches_clean = branches_clean.drop_duplicates('branch_id').reset_index(drop=True)
branches_clean['branch_state'] = canonical_labels(branches_clean['branch_state'])

tickets_clean = clean_ids(tickets_df.copy(), ['ticket_id', 'request_id', 'customer_id'])
tickets_clean = tickets_clean.drop_duplicates('ticket_id').reset_index(drop=True)
tickets_clean['ticket_status'] = canonical_labels(tickets_clean['ticket_status'])
tickets_clean['assigned_team'] = canonical_labels(tickets_clean['assigned_team'])
tickets_clean['escalation_flag'] = to_bool(tickets_clean['escalation_flag'])
tickets_clean['resolution_time_hours'] = pd.to_numeric(tickets_clean['resolution_time_hours'], errors='coerce')
tickets_clean['ticket_created_at'] = pd.to_datetime(tickets_clean['ticket_created_date'], format='%Y-%m-%d', errors='coerce')

updates_clean = clean_ids(updates_df.copy(), ['update_id', 'request_id'])
updates_clean = updates_clean.drop_duplicates('update_id').reset_index(drop=True)
for c in ['status_from', 'status_to', 'updated_by_role']:
    updates_clean[c] = canonical_labels(updates_clean[c])
updates_clean['reassignment_flag'] = to_bool(updates_clean['reassignment_flag'])
updates_clean['update_at'] = pd.to_datetime(updates_clean['update_date'], format='%Y-%m-%d', errors='coerce')

complaints_clean = clean_ids(complaints_df.copy(), ['complaint_id', 'request_id', 'customer_id'])
complaints_clean = complaints_clean.drop_duplicates('complaint_id').reset_index(drop=True)
for c in ['complaint_status', 'sentiment_tag', 'complaint_type']:
    complaints_clean[c] = canonical_labels(complaints_clean[c])
complaints_clean['complaint_at'] = pd.to_datetime(complaints_clean['complaint_date'], format='%Y-%m-%d', errors='coerce')

# Task 04.3: broken foreign keys and bad dates
valid_requests = set(requests_clean['request_id'])
fk_report = pd.DataFrame([
    ('tickets_clean',    len(tickets_clean),    int((~tickets_clean['request_id'].isin(valid_requests)).sum()),    int(tickets_clean['ticket_created_at'].isna().sum())),
    ('updates_clean',    len(updates_clean),    int((~updates_clean['request_id'].isin(valid_requests)).sum()),    int(updates_clean['update_at'].isna().sum())),
    ('complaints_clean', len(complaints_clean), int((~complaints_clean['request_id'].isin(valid_requests)).sum()), int(complaints_clean['complaint_at'].isna().sum())),
], columns=['table', 'rows', 'request_id_not_in_requests', 'unparseable_dates'])
display(fk_report)
print('Requests whose customer_id is not in customers_clean:', int((~requests_clean['customer_id'].isin(customers_clean['customer_id'])).sum()))
print('Requests whose branch_id is not in branches_clean   :', int((~requests_clean['branch_id'].isin(branches_clean['branch_id'])).sum()))
print('Customer segments:', customers_clean['customer_segment'].value_counts().to_dict())
print('Ticket statuses  :', tickets_clean['ticket_status'].value_counts().to_dict())

,table,rows,request_id_not_in_requests,unparseable_dates
0,tickets_clean,1500,31,6
1,updates_clean,1300,30,5
2,complaints_clean,1500,29,8


Requests whose customer_id is not in customers_clean: 14
Requests whose branch_id is not in branches_clean   : 11
Customer segments: {'Student': 184, 'Imperia': 182, 'Senior Citizen': 179, 'NRI': 169, 'Preferred': 168, 'Corporate': 160, 'Classic': 158}
Ticket statuses  : {'Open': 268, 'Reopened': 258, 'Escalated': 248, 'Closed': 246, 'Resolved': 244, 'Pending': 236}


## Milestone 05 · Measure SLA reliability

**From the team:** Management needs reproducible breached and still-pending cases, not artificial breaches caused by using a later system clock.

**Your responsibility:** Use an explicit reporting snapshot derived from the source dates, calculate elapsed hours from request creation through resolution or snapshot, flag completed vs open breaches, and summarize category trends.

**Handover evidence:** `sla_df` with `elapsed_hours`, `sla_breached`, `open_sla_breach`, `sla_date_issue`, plus `category_sla_summary`.

In [ ]:
# Milestone 05: Measure SLA reliability
# Task 05.1: Choose and display a dataset-derived snapshot timestamp; do not use the current date.
# Task 05.2: Compute elapsed_hours only for valid records; calculate SLA breached/open breach flags.
# Task 05.3: Produce category_sla_summary with counts and rates.
# Implement your solution below and display the required handover evidence.

# Task 05.1: snapshot = the latest date that appears anywhere in the request data (end of that day)
latest_date = max(requests_clean['created_at'].max(),
                  requests_clean['last_updated_at'].max(),
                  requests_clean['resolved_at'].max())
snapshot_ts = latest_date.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
print('Reporting snapshot (from the data, not today):', snapshot_ts)

# Task 05.2: resolved requests stop the clock at resolved_at, open requests at the snapshot
sla_df = requests_clean[['request_id', 'request_category', 'request_status', 'is_resolved',
                         'created_at', 'resolved_at', 'sla_hours', 'request_date_issue']].copy()
end_time = sla_df['resolved_at'].where(sla_df['is_resolved'], snapshot_ts)
raw_elapsed = (end_time - sla_df['created_at']).dt.total_seconds() / 3600

# a record is unusable for SLA maths if its dates are broken, the time is negative, or the SLA is missing
sla_df['sla_date_issue'] = sla_df['request_date_issue'] | raw_elapsed.isna() | (raw_elapsed < 0) | sla_df['sla_hours'].isna()
sla_df['elapsed_hours'] = raw_elapsed.where(~sla_df['sla_date_issue'])           # blank for invalid records, never guessed
sla_df['sla_breached'] = sla_df['elapsed_hours'].gt(sla_df['sla_hours'])         # elapsed more than allowed
sla_df['open_sla_breach'] = sla_df['sla_breached'] & ~sla_df['is_resolved']       # breached AND still not resolved

# sanity check against the PRD rule table (request-level sla_hours should agree)
SLA_RULES = {'Account': 48, 'Card': 36, 'Loan': 96, 'KYC': 72, 'Complaint': 72, 'Digital Banking': 24}
mismatch = (sla_df['request_category'].map(SLA_RULES) != sla_df['sla_hours']).sum()
print('Requests whose sla_hours differs from the category rule:', int(mismatch))
print('SLA date issues (excluded from breach maths):', int(sla_df['sla_date_issue'].sum()))
print('Breached:', int(sla_df['sla_breached'].sum()), '| of which still open:', int(sla_df['open_sla_breach'].sum()))

# Task 05.3
category_sla_summary = (sla_df.groupby('request_category')
    .agg(total_requests=('request_id', 'count'),
         resolved_requests=('is_resolved', 'sum'),
         breached_requests=('sla_breached', 'sum'),
         open_breaches=('open_sla_breach', 'sum'),
         avg_elapsed_hours=('elapsed_hours', 'mean'))
    .reset_index())
category_sla_summary['breach_rate'] = category_sla_summary['breached_requests'] / category_sla_summary['total_requests']
category_sla_summary = category_sla_summary.sort_values('breach_rate', ascending=False).reset_index(drop=True)
category_sla_summary.round(3)

Reporting snapshot (from the data, not today): 2026-06-26 23:59:59
Requests whose sla_hours differs from the category rule: 0
SLA date issues (excluded from breach maths): 11
Breached: 4030 | of which still open: 2628


,request_category,total_requests,resolved_requests,breached_requests,open_breaches,avg_elapsed_hours,breach_rate
0,Digital Banking,841,375,750,466,1256.121,0.892
1,Card,825,387,698,437,1225.067,0.846
2,Account,866,399,699,466,1214.091,0.807
3,KYC,802,386,618,416,1195.399,0.771
4,Complaint,822,407,630,414,1146.472,0.766
5,Loan,836,406,635,429,1120.000,0.760


## Milestone 06 · Analyze branch ownership and customer segments

**From the team:** Branches need their workloads and breach ratios, including requests linked to missing branch metadata.

**Your responsibility:** Join branch/customer attributes with many-to-one validation, analyze city/zone/segment and create branch metrics.

**Handover evidence:** `requests_enriched` retaining all requests; `branch_sla_summary` with total requests, breached count and breach rate.

In [ ]:
# Milestone 06: Analyze branch ownership and customer segments
# Task 06.1: Left join clean branch/customer attributes using validated many-to-one merges.
# Task 06.2: Compute branch_sla_summary with total and breached counts plus breach_rate.
# Task 06.3: Check row reconciliation and unmatched branch/customer joins.
# Implement your solution below and display the required handover evidence.

branch_cols = ['branch_id', 'branch_name', 'branch_city', 'branch_state', 'branch_zone', 'branch_type']
customer_cols = ['customer_id', 'customer_segment', 'account_type', 'relationship_age_days']

# validate='many_to_one': many requests per branch/customer, but each branch/customer must appear once
requests_enriched = (requests_clean
    .merge(sla_df[['request_id', 'elapsed_hours', 'sla_breached', 'open_sla_breach', 'sla_date_issue']],
           on='request_id', how='left', validate='one_to_one')
    .merge(branches_clean[branch_cols], on='branch_id', how='left', validate='many_to_one')
    .merge(customers_clean[customer_cols], on='customer_id', how='left', validate='many_to_one'))
requests_enriched['branch_matched'] = requests_enriched['branch_name'].notna()
requests_enriched['customer_matched'] = requests_enriched['customer_segment'].notna()

# Task 06.3: reconciliation
assert len(requests_enriched) == len(requests_clean), 'Join changed the number of requests!'
assert set(requests_enriched['request_id']) == set(requests_clean['request_id'])
print('Rows before/after joins:', len(requests_clean), '/', len(requests_enriched), '-> reconciled')
print('Requests with unmatched branch  :', int((~requests_enriched['branch_matched']).sum()))
print('Requests with unmatched customer:', int((~requests_enriched['customer_matched']).sum()))

# Task 06.2: branch metrics (requests with unknown branch metadata still keep their own branch_id row)
branch_sla_summary = (requests_enriched.groupby('branch_id')
    .agg(total_requests=('request_id', 'count'),
         breached_requests=('sla_breached', 'sum'),
         open_breaches=('open_sla_breach', 'sum'))
    .reset_index())
branch_sla_summary['breach_rate'] = branch_sla_summary['breached_requests'] / branch_sla_summary['total_requests']
branch_sla_summary = branch_sla_summary.merge(branches_clean[branch_cols], on='branch_id', how='left', validate='one_to_one')
branch_sla_summary['branch_name'] = branch_sla_summary['branch_name'].fillna('Unmatched branch')
branch_sla_summary = branch_sla_summary.sort_values(['open_breaches', 'breached_requests'], ascending=False).reset_index(drop=True)
assert branch_sla_summary['total_requests'].sum() == len(requests_clean)

# city / zone / segment views
zone_summary = requests_enriched.assign(branch_zone=requests_enriched['branch_zone'].fillna('Unmatched')).groupby('branch_zone').agg(
    total_requests=('request_id', 'count'), breached_requests=('sla_breached', 'sum')).assign(
    breach_rate=lambda d: d['breached_requests'] / d['total_requests']).round(3)
segment_summary = requests_enriched.assign(customer_segment=requests_enriched['customer_segment'].fillna('Unmatched')).groupby('customer_segment').agg(
    total_requests=('request_id', 'count'), breached_requests=('sla_breached', 'sum')).assign(
    breach_rate=lambda d: d['breached_requests'] / d['total_requests']).round(3)
city_summary = requests_enriched.groupby('branch_city').agg(total_requests=('request_id', 'count'), breached_requests=('sla_breached', 'sum')).sort_values('breached_requests', ascending=False).head(5)
display(zone_summary); display(segment_summary); display(city_summary)
branch_sla_summary.head(8).round(3)

Rows before/after joins: 4992 / 4992 -> reconciled
Requests with unmatched branch  : 11
Requests with unmatched customer: 14


,total_requests,breached_requests,breach_rate
branch_zone,,,
East,459,371,0.808
North,1593,1279,0.803
South,1950,1575,0.808
Unmatched,11,10,0.909
West,979,795,0.812


,total_requests,breached_requests,breach_rate
customer_segment,,,
Classic,643,516,0.802
Corporate,650,522,0.803
Imperia,767,621,0.810
NRI,705,570,0.809
Preferred,709,578,0.815
Senior Citizen,721,587,0.814
Student,783,626,0.799
Unmatched,14,10,0.714


,total_requests,breached_requests
branch_city,,
Noida,252,211
Hyderabad,247,194
Warangal,220,187
Durgapur,210,163
Ahmedabad,193,152


,branch_id,total_requests,breached_requests,open_breaches,breach_rate,branch_name,branch_city,branch_state,branch_zone,branch_type
0,BR0535,14,11,10,0.786,HDFC Ahmedabad Service Branch 0535,Ahmedabad,Gujarat,West,Digital Service Hub
1,BR0723,14,11,9,0.786,HDFC Surat Service Branch 0723,Surat,Gujarat,West,Urban
2,BR0421,11,11,8,1.000,HDFC Hubballi Service Branch 0421,Hubballi,Karnataka,South,Metro
3,BR0011,11,10,8,0.909,HDFC Howrah Service Branch 0011,Howrah,West Bengal,East,Semi-Urban
4,BR0143,12,10,8,0.833,HDFC Gurugram Service Branch 0143,Gurugram,Delhi NCR,North,Digital Service Hub
5,BR0802,9,8,8,0.889,HDFC Nashik Service Branch 0802,Nashik,Maharashtra,West,Metro
6,BR0096,12,11,7,0.917,HDFC Madurai Service Branch 0096,Madurai,Tamil Nadu,South,Semi-Urban
7,BR0198,12,11,7,0.917,HDFC Salem Service Branch 0198,Salem,Tamil Nadu,South,Digital Service Hub


## Milestone 07 · Investigate complaints, tickets and workflow updates

**From the team:** Escalations may be repeated across several events for a single request; supervisors need totals without duplicate request rows.

**Your responsibility:** Aggregate complaint severity/open status, ticket escalations and pending cases, and reassignment/status-update volumes at request level. Flag orphan events.

**Handover evidence:** `complaints_by_request`, `tickets_by_request`, `updates_by_request`, each unique on `request_id`; an orphan-link summary.

In [ ]:
# Milestone 07: Investigate complaints, tickets and workflow updates
# Task 07.1: Aggregate complaint counts, open complaints, and severe open complaints per request.
# Task 07.2: Aggregate tickets and escalations per request; aggregate status updates and reassignments.
# Task 07.3: Count event records with no matching request; assert unique request IDs in aggregates.
# Implement your solution below and display the required handover evidence.

OPEN_COMPLAINT = ['Open', 'In Review', 'Reopened', 'Escalated']   # complaint not yet settled
SEVERE_SENTIMENT = ['Angry', 'Very Angry']                          # the data has no "critical" tag, so these are "severe"
PENDING_TICKET = ['Open', 'Pending', 'Reopened', 'Escalated']       # ticket still active

# Task 07.1 - complaints
c = complaints_clean.copy()
c['is_open'] = c['complaint_status'].isin(OPEN_COMPLAINT)
c['is_severe_open'] = c['is_open'] & c['sentiment_tag'].isin(SEVERE_SENTIMENT)
complaints_by_request = (c.groupby('request_id')
    .agg(complaint_count=('complaint_id', 'count'),
         open_complaint_count=('is_open', 'sum'),
         severe_open_complaint_count=('is_severe_open', 'sum'))
    .reset_index())

# Task 07.2 - tickets
t = tickets_clean.copy()
t['is_pending'] = t['ticket_status'].isin(PENDING_TICKET)
t['is_escalated'] = t['escalation_flag'] | (t['ticket_status'] == 'Escalated')
tickets_by_request = (t.groupby('request_id')
    .agg(ticket_count=('ticket_id', 'count'),
         escalated_ticket_count=('is_escalated', 'sum'),
         pending_ticket_count=('is_pending', 'sum'))
    .reset_index())

# Task 07.2 - status updates
updates_by_request = (updates_clean.groupby('request_id')
    .agg(status_update_count=('update_id', 'count'),
         reassignment_count=('reassignment_flag', 'sum'))
    .reset_index())

# Task 07.3 - uniqueness and orphans (events whose request_id is not a valid request)
for name, frame in [('complaints_by_request', complaints_by_request), ('tickets_by_request', tickets_by_request), ('updates_by_request', updates_by_request)]:
    assert frame['request_id'].is_unique, f'{name} is not unique on request_id'
orphan_summary = pd.DataFrame({
    'event_table': ['complaints', 'tickets', 'status updates'],
    'events': [len(complaints_clean), len(tickets_clean), len(updates_clean)],
    'orphan_events': [int((~complaints_clean['request_id'].isin(valid_requests)).sum()),
                      int((~tickets_clean['request_id'].isin(valid_requests)).sum()),
                      int((~updates_clean['request_id'].isin(valid_requests)).sum())],
    'requests_with_events': [int(complaints_by_request['request_id'].isin(valid_requests).sum()),
                             int(tickets_by_request['request_id'].isin(valid_requests).sum()),
                             int(updates_by_request['request_id'].isin(valid_requests).sum())],
})
display(orphan_summary)
print('Aggregates are unique on request_id. Orphan events are counted here but will not appear in the request-level view.')
print('Requests with repeated complaints (2+):', int((complaints_by_request['complaint_count'] >= 2).sum()))
complaints_by_request.sort_values('complaint_count', ascending=False).head(3)

,event_table,events,orphan_events,requests_with_events
0,complaints,1500,29,1276
1,tickets,1500,31,1269
2,status updates,1300,30,1110


Aggregates are unique on request_id. Orphan events are counted here but will not appear in the request-level view.
Requests with repeated complaints (2+): 181


,request_id,complaint_count,open_complaint_count,severe_open_complaint_count
53,REQ0000228,4,3,1
916,REQ0003642,3,3,2
949,REQ0003793,3,2,0


## Milestone 08 · Build the consolidated request view

**From the team:** Agents should be able to investigate each service request without manually joining six files.

**Your responsibility:** Detect repeated customer/category requests; left-join aggregated event evidence to enriched requests, convert absent counts to zero, and verify row and key reconciliation.

**Handover evidence:** `final_requests_df` with one row per request and `repeat_customer_category` (boolean); all original request IDs retained.

In [ ]:
# Milestone 08: Build the consolidated request view
# Task 08.1: Compute a repeated customer/category request indicator using distinct requests.
# Task 08.2: Left join aggregated event tables to requests_enriched; fill absent event counts with zero.
# Task 08.3: Assert exactly one row per request and full request-ID reconciliation.
# Implement your solution below and display the required handover evidence.

# Task 08.1: the same customer raising 2+ distinct requests in the same category
requests_enriched['repeat_customer_category'] = (
    requests_enriched.groupby(['customer_id', 'request_category'])['request_id'].transform('nunique') > 1)

# Task 08.2: aggregates already have one row per request_id, so the left joins cannot multiply rows
final_requests_df = (requests_enriched
    .merge(complaints_by_request, on='request_id', how='left', validate='one_to_one')
    .merge(tickets_by_request, on='request_id', how='left', validate='one_to_one')
    .merge(updates_by_request, on='request_id', how='left', validate='one_to_one'))
event_count_cols = ['complaint_count', 'open_complaint_count', 'severe_open_complaint_count',
                    'ticket_count', 'escalated_ticket_count', 'pending_ticket_count',
                    'status_update_count', 'reassignment_count']
final_requests_df[event_count_cols] = final_requests_df[event_count_cols].fillna(0).astype(int)

# Task 08.3: reconciliation
assert final_requests_df['request_id'].is_unique
assert set(final_requests_df['request_id']) == set(requests_clean['request_id'])
print('Rows in requests_clean:', len(requests_clean), '| rows in final_requests_df:', len(final_requests_df))
for label, col, src in [('complaints', 'complaint_count', complaints_clean), ('tickets', 'ticket_count', tickets_clean), ('updates', 'status_update_count', updates_clean)]:
    in_view = int(final_requests_df[col].sum())
    matched = int(src['request_id'].isin(valid_requests).sum())
    print(f'{label:10s}: events matched to a request = {matched} | sum in final view = {in_view} | equal: {in_view == matched}')
print('Requests flagged as repeat customer/category:', int(final_requests_df['repeat_customer_category'].sum()))
final_requests_df[['request_id', 'customer_id', 'request_category', 'repeat_customer_category', 'complaint_count', 'ticket_count', 'status_update_count']].head()

Rows in requests_clean: 4992 | rows in final_requests_df: 4992
complaints: events matched to a request = 1471 | sum in final view = 1471 | equal: True
tickets   : events matched to a request = 1469 | sum in final view = 1469 | equal: True
updates   : events matched to a request = 1270 | sum in final view = 1270 | equal: True
Requests flagged as repeat customer/category: 2484


,request_id,customer_id,request_category,repeat_customer_category,complaint_count,ticket_count,status_update_count
0,REQ0000009,CUST001111,Complaint,True,0,0,0
1,REQ0000010,CUST000649,Card,True,0,0,0
2,REQ0000011,CUST000817,Digital Banking,False,0,0,1
3,REQ0000012,CUST000212,Complaint,True,0,1,0
4,REQ0000013,CUST001137,Account,True,0,0,0


## Milestone 09 · Create transparent service triage

**From the team:** Supervisors need an explainable triage queue without a black-box model.

**Your responsibility:** Create independent signals for SLA breach, open/severe complaint, escalation and repeated customer/category requests; assign High for 2+ signals, Medium for 1, Low otherwise. Generate a practical recommended action.

**Handover evidence:** `final_requests_df` with `priority_score`, `request_priority` (High/Medium/Low) and `recommended_action`; show priority counts.

In [ ]:
# Milestone 09: Create transparent service triage
# Task 09.1: Implement four independent signal columns and priority_score.
# Task 09.2: Assign High if score>=2, Medium if score==1, Low otherwise.
# Task 09.3: Create recommended_action and display the priority distribution.
# Implement your solution below and display the required handover evidence.

# Task 09.1: four independent yes/no signals
final_requests_df['signal_sla_breach'] = final_requests_df['sla_breached'].astype(bool)
final_requests_df['signal_severe_open_complaint'] = final_requests_df['severe_open_complaint_count'] > 0
final_requests_df['signal_escalated_ticket'] = final_requests_df['escalated_ticket_count'] > 0
final_requests_df['signal_repeat_request'] = final_requests_df['repeat_customer_category'].astype(bool)
signal_cols = ['signal_sla_breach', 'signal_severe_open_complaint', 'signal_escalated_ticket', 'signal_repeat_request']
final_requests_df['priority_score'] = final_requests_df[signal_cols].sum(axis=1)

# Task 09.2
def score_to_priority(score):
    if score >= 2:
        return 'High'
    if score == 1:
        return 'Medium'
    return 'Low'
final_requests_df['request_priority'] = final_requests_df['priority_score'].map(score_to_priority)

# Task 09.3: a plain-language next step that depends on the priority and on which signals fired
def recommend(row):
    if row['request_priority'] == 'High':
        parts = []
        if row['signal_escalated_ticket']:
            parts.append('call the customer and resolve the escalated ticket')
        if row['signal_severe_open_complaint']:
            parts.append('handle the unhappy customer through the grievance desk')
        if row['signal_sla_breach'] and not row['is_resolved']:
            parts.append('assign an owner and close the overdue request today')
        if not parts:
            parts.append('review the full history and decide the next step today')
        return 'Escalate to supervisor: ' + '; '.join(parts) + '.'
    if row['request_priority'] == 'Medium':
        if row['signal_sla_breach'] and row['is_resolved']:
            return 'Closed after the deadline: review the cause, no customer action needed.'
        if row['signal_sla_breach']:
            return 'Follow up within 24 hours: request is past its SLA.'
        if row['signal_repeat_request']:
            return 'Check why the customer raised this type of request again.'
        if row['signal_escalated_ticket']:
            return 'Monitor the escalated ticket until it is closed.'
        return 'Contact the customer and check the open complaint.'
    return 'No action needed. Routine monitoring only.'
final_requests_df['recommended_action'] = final_requests_df.apply(recommend, axis=1)

PRIORITY_ORDER = ['High', 'Medium', 'Low']
priority_counts = (final_requests_df['request_priority'].value_counts().reindex(PRIORITY_ORDER).rename_axis('request_priority')
                   .reset_index(name='requests'))
priority_counts['share_pct'] = (priority_counts['requests'] / len(final_requests_df) * 100).round(1)
display(priority_counts)
print('Signal counts:', final_requests_df[signal_cols].sum().to_dict())
final_requests_df[['request_id', 'priority_score', 'request_priority', 'recommended_action']].head()

,request_priority,requests,share_pct
0,High,2525,50.6
1,Medium,2071,41.5
2,Low,396,7.9


Signal counts: {'signal_sla_breach': 4030, 'signal_severe_open_complaint': 356, 'signal_escalated_ticket': 683, 'signal_repeat_request': 2484}


,request_id,priority_score,request_priority,recommended_action
0,REQ0000009,2,High,Escalate to supervisor: assign an owner and cl...
1,REQ0000010,2,High,Escalate to supervisor: assign an owner and cl...
2,REQ0000011,0,Low,No action needed. Routine monitoring only.
3,REQ0000012,2,High,Escalate to supervisor: assign an owner and cl...
4,REQ0000013,2,High,Escalate to supervisor: review the full histor...


## Milestone 10 · Hand over operational reports

**From the team:** The service lead needs a readable request queue and branch metrics without running Python.

**Your responsibility:** Create ordered detailed/priority/branch reports; export CSV and Excel; write 3–5 evidence-based observations and acknowledge dataset limitations.

**Handover evidence:** `final_report`, `priority_summary`, `branch_sla_summary`; export `hdfc_request_resolution_report.csv`, `hdfc_priority_summary.csv`, `hdfc_branch_sla_summary.xlsx`.

In [ ]:
# Milestone 10: Hand over operational reports
# Task 10.1: Build final_report sorted High > Medium > Low and priority_summary.
# Task 10.2: Write two CSV exports and one Excel branch summary into hdfc_reports/.
# Task 10.3: Display key totals and write evidence-based findings below.
# Implement your solution below and display the required handover evidence.

# Task 10.1
report_cols = ['request_id', 'request_priority', 'priority_score', 'recommended_action',
               'request_category', 'request_subcategory', 'channel', 'request_status', 'assigned_team',
               'created_at', 'resolved_at', 'sla_hours', 'elapsed_hours', 'sla_breached', 'open_sla_breach',
               'reported_amount', 'customer_id', 'customer_segment', 'branch_id', 'branch_name', 'branch_city', 'branch_zone',
               'complaint_count', 'severe_open_complaint_count', 'ticket_count', 'escalated_ticket_count',
               'status_update_count', 'reassignment_count', 'repeat_customer_category', 'sla_date_issue']
final_report = final_requests_df[report_cols].copy()
final_report['elapsed_hours'] = final_report['elapsed_hours'].round(1)
final_report['request_priority'] = pd.Categorical(final_report['request_priority'], categories=PRIORITY_ORDER, ordered=True)
final_report = final_report.sort_values(['request_priority', 'priority_score', 'elapsed_hours'],
                                        ascending=[True, False, False]).reset_index(drop=True)
final_report['request_priority'] = final_report['request_priority'].astype(str)

priority_summary = (final_requests_df.groupby('request_priority')
    .agg(total_requests=('request_id', 'count'),
         breached_requests=('sla_breached', 'sum'),
         open_requests=('is_resolved', lambda s: int((~s).sum())),
         avg_priority_score=('priority_score', 'mean'))
    .reindex(PRIORITY_ORDER).reset_index())
priority_summary['share_pct'] = (priority_summary['total_requests'] / priority_summary['total_requests'].sum() * 100).round(1)

# Task 10.2: exports (REPORT_DIR is also used by the assessment cell)
REPORT_DIR = 'hdfc_reports'
Path(REPORT_DIR).mkdir(exist_ok=True)
final_report.to_csv(Path(REPORT_DIR) / 'hdfc_request_resolution_report.csv', index=False)
priority_summary.to_csv(Path(REPORT_DIR) / 'hdfc_priority_summary.csv', index=False)
branch_sla_summary.to_excel(Path(REPORT_DIR) / 'hdfc_branch_sla_summary.xlsx', index=False)
print('Files in', REPORT_DIR + '/:', sorted(p.name for p in Path(REPORT_DIR).iterdir()))

# Task 10.3: key totals
print('\nKEY TOTALS')
print('Requests               :', len(final_report))
print('Resolved / Closed      :', int(final_requests_df['is_resolved'].sum()))
print('SLA breached           :', int(final_requests_df['sla_breached'].sum()), f"({final_requests_df['sla_breached'].mean():.1%})")
print('Open and past SLA      :', int(final_requests_df['open_sla_breach'].sum()))
print('Valid complaints       :', int(final_requests_df['complaint_count'].sum()), '| severe and open:', int(final_requests_df['severe_open_complaint_count'].sum()))
print('Valid tickets          :', int(final_requests_df['ticket_count'].sum()), '| escalated:', int(final_requests_df['escalated_ticket_count'].sum()))
display(priority_summary.round(2))
final_report.head(5)

Files in hdfc_reports/: ['hdfc_branch_sla_summary.xlsx', 'hdfc_priority_summary.csv', 'hdfc_request_resolution_report.csv']

KEY TOTALS
Requests               : 4992
Resolved / Closed      : 2360
SLA breached           : 4030 (80.7%)
Open and past SLA      : 2628
Valid complaints       : 1471 | severe and open: 366
Valid tickets          : 1469 | escalated: 733


,request_priority,total_requests,breached_requests,open_requests,avg_priority_score,share_pct
0,High,2525,2411,1575,2.17,50.6
1,Medium,2071,1619,1057,1.00,41.5
2,Low,396,0,0,0.00,7.9


,request_id,request_priority,priority_score,recommended_action,request_category,request_subcategory,channel,request_status,assigned_team,created_at,...,branch_city,branch_zone,complaint_count,severe_open_complaint_count,ticket_count,escalated_ticket_count,status_update_count,reassignment_count,repeat_customer_category,sla_date_issue
0,REQ0004986,High,4,Escalate to supervisor: call the customer and ...,Loan,Loan Statement,Email,In Progress,Retail Assets,2026-01-01 03:13:00,...,Kozhikode,South,1,1,1,1,1,1,True,False
1,REQ0000407,High,4,Escalate to supervisor: call the customer and ...,Digital Banking,UPI Issue,NetBanking,Escalated,Digital Banking Support,2026-01-12 04:21:00,...,Kochi,South,2,1,1,1,0,0,True,False
2,REQ0002992,High,4,Escalate to supervisor: call the customer and ...,Complaint,Service Delay,Call Center,Escalated,Customer Grievance,2026-01-27 02:17:00,...,New Delhi,North,1,1,2,1,0,0,True,False
3,REQ0001993,High,4,Escalate to supervisor: call the customer and ...,Card,Failed ATM Withdrawal,WhatsApp,Open,Cards Operations,2026-02-22 22:59:00,...,Kochi,South,1,1,1,1,0,0,True,False
4,REQ0004244,High,4,Escalate to supervisor: call the customer and ...,Loan,EMI Issue,Branch,In Progress,Retail Assets,2026-03-03 18:19:00,...,Kolkata,East,1,1,1,1,0,0,True,False


## Handover notes · What should the Support Lead know?

Write 3–5 observations based on numbers in your actual outputs. Include one operational implication and at least one data-quality limitation.

**Your findings:**

1. **Most requests miss their SLA.** At the snapshot date (26 June 2026), 4,030 of 4,992 requests (80.7%) were breached, and 2,628 of them are still open and past their deadline. Digital Banking has the highest breach rate (89.2%) and Loan the lowest (76.0%).
2. **The problem is everywhere, not in one place.** Breach rates are almost the same across zones (80.3% to 81.2%) and customer segments (79.9% to 81.5%). The worst branch (BR0535, Ahmedabad) has only 10 open breaches. This points to a process or capacity issue across the whole operation, not one weak branch.
3. **Half the queue is High priority.** Priority split: High 2,525 (50.6%), Medium 2,071 (41.5%), Low 396 (7.9%). Operational implication: this is too many to treat equally, because the breach signal fires on 4,030 requests and the repeat-request signal on 2,484. Supervisors should start with score 3 to 4 cases (breached, escalated ticket and severe open complaint together), which the report sorts to the top.
4. **Complaints and tickets are often unresolved.** Of 1,471 valid complaints, 366 are both severe (Angry or Very Angry) and still open. 181 requests have 2 or more complaints, and 733 of 1,469 valid tickets are escalated.
5. **Data-quality limitations.** 8 duplicate request IDs were removed; 90 complaint, ticket and update records point to requests that do not exist; 11 requests have unusable dates and are excluded from SLA maths; 14 requests have no matching customer and 11 no matching branch. Also, many open requests are months old at the snapshot date, so breach counts for open items are high by design, and `status_notes.txt` was not supplied.

**Assumptions and choices**
* Reporting snapshot = the latest date found in the request data (not today's date), so results are reproducible.
* A resolved request is measured until the end of its resolution date, because the data has a date only.
* Duplicate request IDs: the first (original) row is kept. Invalid dates are flagged and excluded from SLA maths, never deleted.
* "Severe" complaint = Angry or Very Angry sentiment (the data has no "critical" tag). Open complaint = Open, In Review, Reopened or Escalated.
* Escalated ticket = escalation flag Yes, or ticket status Escalated.
* `status_notes.txt` is not part of the three-file package, so it was not used.


## Final milestone · Automated assessment

**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose Runtime → Restart session, then Run all before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved .ipynb in a fresh kernel with clean input files.

Run the self-check after all ten milestones. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does not require an answer key to be shared with students.

**How to read results:** Pass means the available output meets the named structural or business-rule check; Review means something is absent or differs from the expected contract. This is formative self-checking, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [ ]:
# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({'Milestone': task, 'Check': criterion, 'Result': 'Pass' if passed else 'Review', 'Points': 5 if passed else 0, 'Feedback': msg})

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(col in frame and pd.api.types.is_datetime64_any_dtype(frame[col]) for col in cols)

def expected_counts(name, count_col, source, key):
    a = df(name)
    s = df(source)
    return (has_cols(name,[key,count_col]) and unique_nonnull(a,key)
            and numeric(a,count_col)
            and a[count_col].sum()==s[key].notna().sum())

def event_counts_match():
    m = df('final_requests_df')
    if not unique_nonnull(m,'request_id'): return False
    for source,col in [('complaints_clean','complaint_count'),
                       ('tickets_clean','ticket_count'),
                       ('updates_clean','status_update_count')]:
        s = df(source)
        if col not in m or 'request_id' not in s: return False
        counts = s[s.request_id.isin(m.request_id)].groupby('request_id').size()
        actual = m.set_index('request_id')[col].fillna(0)
        aligned = counts.reindex(actual.index,fill_value=0)
        if not actual.eq(aligned).all(): return False
    return True

def sla_matches():
    s = df('sla_df')
    required = ['request_id','elapsed_hours','sla_hours',
                'sla_breached','open_sla_breach','sla_date_issue']
    if not has_cols('sla_df',required): return False
    if not numeric(s,'elapsed_hours') or not numeric(s,'sla_hours'): return False
    if any(not pd.api.types.is_bool_dtype(s[x]) for x in
           ['sla_breached','open_sla_breach']): return False
    expected = (s.elapsed_hours.gt(s.sla_hours)
                & s.elapsed_hours.notna() & s.sla_hours.notna())
    return (unique_nonnull(s,'request_id')
            and s.sla_breached.eq(expected).all()
            and (~s.open_sla_breach | s.sla_breached).all())

def signals_match():
    m = df('final_requests_df')
    flags = ['signal_sla_breach','signal_severe_open_complaint',
             'signal_escalated_ticket','signal_repeat_request']
    if not has_cols('final_requests_df',flags+['priority_score']): return False
    if any(not pd.api.types.is_bool_dtype(m[x]) for x in flags): return False
    return (numeric(m,'priority_score')
            and m.priority_score.eq(m[flags].sum(axis=1)).all())

def priorities_match():
    m = df('final_requests_df')
    if not has_cols('final_requests_df',
                    ['priority_score','request_priority','recommended_action']):
        return False
    expected = m.priority_score.map(
        lambda x: 'High' if x>=2 else ('Medium' if x==1 else 'Low'))
    return (m.request_priority.eq(expected).all()
            and m.recommended_action.notna().all()
            and m.recommended_action.astype(str).str.strip().ne('').all())

def report_matches():
    r = df('final_report')
    m = df('final_requests_df')
    if not has_cols('final_report',['request_id','request_priority']): return False
    return (unique_nonnull(r,'request_id')
            and len(r)==len(m)
            and set(r.request_id)==set(m.request_id)
            and r.set_index('request_id').request_priority.eq(
                m.set_index('request_id').loc[r.request_id,'request_priority'].to_numpy()).all())

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary',['request_priority','total_requests']): return False
    expected = df('final_requests_df').request_priority.value_counts()
    got = s.groupby('request_priority').total_requests.sum()
    return (got.reindex(expected.index,fill_value=0).eq(expected).all()
            and got.sum()==len(df('final_requests_df')))

check(1,'Service requests loaded, key fields present',lambda: has_cols('requests_df',['request_id','customer_id','branch_id']))
check(1,'Service request source is non-empty',lambda: len(df('requests_df'))>0 and df('requests_df').request_id.notna().all())
check(2,'Five supporting DataFrames loaded',lambda: all(isinstance(df(n),pd.DataFrame) for n in ['customers_df','branches_df','tickets_df','updates_df','complaints_df']))
check(2,'Supporting event identifiers present',lambda: all(has_cols(n,[k,'request_id']) for n,k in [('tickets_df','ticket_id'),('updates_df','update_id'),('complaints_df','complaint_id')]))
check(3,'One clean record per service request',lambda: unique_nonnull(df('requests_clean'),'request_id') and len(df('requests_clean'))==df('requests_df').request_id.nunique())
check(3,'Request dates, SLA and anomaly flags prepared',lambda: real_dates(df('requests_clean'),['created_at']) and numeric(df('requests_clean'),'sla_hours') and has_cols('requests_clean',['request_date_issue']))
check(4,'Customer and branch dimension keys are unique',lambda: unique_nonnull(df('customers_clean'),'customer_id') and unique_nonnull(df('branches_clean'),'branch_id'))
check(4,'Complaint, ticket and update IDs are unique',lambda: all(unique_nonnull(df(n),k) for n,k in [('complaints_clean','complaint_id'),('tickets_clean','ticket_id'),('updates_clean','update_id')]))
check(5,'SLA calculations follow business rules',sla_matches)
check(5,'Category SLA summary reconciles to requests',lambda: has_cols('category_sla_summary',['total_requests']) and df('category_sla_summary').total_requests.sum()==len(df('sla_df')))
check(6,'Enriched requests preserve request population',lambda: unique_nonnull(df('requests_enriched'),'request_id') and set(df('requests_enriched').request_id)==set(df('requests_clean').request_id))
check(6,'Branch SLA summary reconciles',lambda: has_cols('branch_sla_summary',['total_requests','breach_rate']) and df('branch_sla_summary').total_requests.sum()==len(df('requests_clean')) and df('branch_sla_summary').breach_rate.between(0,1).all())
check(7,'Event aggregates preserve unique request keys',lambda: all(unique_nonnull(df(n),'request_id') for n in ['complaints_by_request','tickets_by_request','updates_by_request']))
check(7,'Aggregated event totals match cleaned sources',lambda: all([expected_counts('complaints_by_request','complaint_count','complaints_clean','request_id'),expected_counts('tickets_by_request','ticket_count','tickets_clean','request_id'),expected_counts('updates_by_request','status_update_count','updates_clean','request_id')]))
check(8,'Master table contains each request exactly once',lambda: unique_nonnull(df('final_requests_df'),'request_id') and set(df('final_requests_df').request_id)==set(df('requests_clean').request_id))
check(8,'Master event counts reconcile',lambda: event_counts_match() and has_cols('final_requests_df',['repeat_customer_category']))
check(9,'Priority signals and scores are consistent',signals_match)
check(9,'Priority classification and actions are consistent',priorities_match)
check(10,'Detailed report and export exist',lambda: report_matches() and (Path(REPORT_DIR)/'hdfc_request_resolution_report.csv').is_file())
check(10,'Priority and branch summaries exported',lambda: summary_matches() and all((Path(REPORT_DIR)/n).is_file() for n in ['hdfc_priority_summary.csv','hdfc_branch_sla_summary.xlsx']))

assessment_df = pd.DataFrame(_checks)
score_by_task = assessment_df.groupby('Milestone',as_index=False).agg(Points=('Points','sum'),Passed=('Result',lambda x:(x=='Pass').sum()))
score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({0:'Review',1:'Partially verified',2:'Checks passed'})
print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone','Points','Maximum','Status']])
print(f'Automated checks score: {assessment_df.Points.sum()}/100')
print('Checks needing attention:')
display(assessment_df.loc[assessment_df.Result=='Review',['Milestone','Check','Feedback']].reset_index(drop=True))

# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json
print('HDFC_ASSESSMENT_JSON=' + _grade_json.dumps({'points': int(assessment_df.Points.sum()), 'milestones': score_by_task[['Milestone','Points','Maximum','Status']].to_dict('records'), 'checks': assessment_df[['Milestone','Check','Result','Points']].to_dict('records')}))

AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


HDFC_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Service requests loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Service request source is non-empty", "Result": "Pass", "Points

## Submission handover

* Submit your completed student notebook (.ipynb) with the three exported reports and visible assessment output (or share a Colab copy if required).
* Include 3–5 supported findings and a short explanation of AI usage, assumptions, and unresolved data-quality issues.
* The ZIP includes six structured business datasets plus `status_notes.txt`; never substitute datasets from another project.
* Student and instructor notebooks should be run from a fresh session with the supplied original data files.